In [2]:
#Import Section
import pandas as pd
import sqlite3

In [3]:
#Data Download
df = pd.read_csv("Bank Customer Churn Prediction.csv")

In [5]:
#Make Database
conn = sqlite3.connect('bank_churn.db')

In [15]:
#load data into table
df.to_sql('bank_customers', conn, if_exists='replace', index=False)

10000

In [12]:
print("تم انشاء قاعدة البيانات بنجاح")
print(f"عدد الصفوف: {len(df)}")
print(f"الأعمدة: {df.columns.tolist()}")


تم انشاء قاعدة البيانات بنجاح
عدد الصفوف: 10000
الأعمدة: ['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary', 'churn']


In [13]:
#Check on data
cursor = conn.cursor()

In [16]:
#first 5 rows
result = pd.read_sql_query("SELECT * FROM bank_customers LIMIT 5", conn)
print(result)

   customer_id  credit_score country  gender  age  tenure    balance  \
0     15634602           619  France  Female   42       2       0.00   
1     15647311           608   Spain  Female   41       1   83807.86   
2     15619304           502  France  Female   42       8  159660.80   
3     15701354           699  France  Female   39       1       0.00   
4     15737888           850   Spain  Female   43       2  125510.82   

   products_number  credit_card  active_member  estimated_salary  churn  
0                1            1              1         101348.88      1  
1                1            0              1         112542.58      0  
2                3            1              0         113931.57      1  
3                2            0              0          93826.63      0  
4                1            1              1          79084.10      0  


In [22]:
# Total customers and churn rate
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
"""
result = pd.read_sql_query(query, conn)
print(result)

   total_customers  churned_customers  churn_rate_percent
0            10000               2037               20.37


In [25]:
# Customer distribution by country
query = """
SELECT
    country,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
GROUP BY country
ORDER BY total_customers DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

   country  total_customers  churned_customers  churn_rate_percent
0   France             5014                810               16.15
1  Germany             2509                814               32.44
2    Spain             2477                413               16.67


In [26]:
# Churn rate by gender
query = """
SELECT
    gender,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
GROUP BY gender
ORDER BY churn_rate_percent DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

   gender  total_customers  churned_customers  churn_rate_percent
0  Female             4543               1139               25.07
1    Male             5457                898               16.46


In [27]:
# Average balance and salary: churned vs retained
query = """
SELECT
    churn,
    COUNT(*) AS total_customers,
    ROUND(AVG(balance), 2) AS avg_balance,
    ROUND(AVG(estimated_salary), 2) AS avg_salary,
    ROUND(AVG(credit_score), 2) AS avg_credit_score
FROM bank_customers
GROUP BY churn
"""
result = pd.read_sql_query(query, conn)
print(result)

   churn  total_customers  avg_balance  avg_salary  avg_credit_score
0      0             7963     72745.30    99738.39            651.85
1      1             2037     91108.54   101465.68            645.35


In [28]:
# Churn rate by number of products
query = """
SELECT
    products_number,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
GROUP BY products_number
ORDER BY products_number
"""
result = pd.read_sql_query(query, conn)
print(result)

   products_number  total_customers  churned_customers  churn_rate_percent
0                1             5084               1409               27.71
1                2             4590                348                7.58
2                3              266                220               82.71
3                4               60                 60              100.00


In [29]:
# Churn rate by number of products
query = """
SELECT
    products_number,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
GROUP BY products_number
ORDER BY products_number
"""
result = pd.read_sql_query(query, conn)
print(result)

   products_number  total_customers  churned_customers  churn_rate_percent
0                1             5084               1409               27.71
1                2             4590                348                7.58
2                3              266                220               82.71
3                4               60                 60              100.00


In [30]:
# Churn rate by age group
query = """
SELECT
    CASE
        WHEN age < 30 THEN 'Under 30'
        WHEN age BETWEEN 30 AND 40 THEN '30-40'
        WHEN age BETWEEN 41 AND 50 THEN '41-50'
        WHEN age BETWEEN 51 AND 60 THEN '51-60'
        ELSE 'Over 60'
    END AS age_group,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
GROUP BY age_group
ORDER BY churn_rate_percent DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

  age_group  total_customers  churned_customers  churn_rate_percent
0     51-60              797                448               56.21
1     41-50             2320                788               33.97
2   Over 60              464                115               24.78
3     30-40             4778                562               11.76
4  Under 30             1641                124                7.56


In [31]:
# High-balance customers who churned (above average balance)
query = """
SELECT
    COUNT(*) AS high_balance_churned,
    ROUND(AVG(balance), 2) AS avg_balance,
    ROUND(AVG(age), 2) AS avg_age
FROM bank_customers
WHERE churn = 1
AND balance > (SELECT AVG(balance) FROM bank_customers)
"""
result = pd.read_sql_query(query, conn)
print(result)

   high_balance_churned  avg_balance  avg_age
0                  1426    125504.86    44.76


In [32]:
# Country with highest churn among active members
query = """
SELECT
    country,
    COUNT(*) AS active_customers,
    SUM(churn) AS churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
FROM bank_customers
WHERE active_member = 1
GROUP BY country
HAVING churn_rate_percent = (
    SELECT MAX(churn_rate_percent)
    FROM (
        SELECT ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS churn_rate_percent
        FROM bank_customers
        WHERE active_member = 1
        GROUP BY country
    )
)
"""
result = pd.read_sql_query(query, conn)
print(result)

   country  active_customers  churned  churn_rate_percent
0  Germany              1248      296               23.72


In [33]:
# Save all results to a summary file
summary = """
# Bank Customer Churn Analysis — Key Findings

## Dataset
- Total Customers: 10,000
- Features: 12 (credit score, country, gender, age, balance, products, churn, etc.)

## Key Findings

### 1. Overall Churn Rate
- 20.37% of customers churned (2,037 out of 10,000)

### 2. Churn by Country
- Germany: 32.44% — highest churn rate
- Spain: 16.67%
- France: 16.15% — most loyal

### 3. Churn by Gender
- Female: 25.07% churn rate
- Male: 16.46% churn rate

### 4. Churn by Number of Products
- 1 product: 27.71%
- 2 products: 7.58% — most loyal segment
- 3 products: 82.71%
- 4 products: 100% — all churned

### 5. Churn by Age Group
- 51-60: 56.21% — highest risk
- 41-50: 33.97%
- Over 60: 24.78%
- 30-40: 11.76%
- Under 30: 7.56% — most loyal

### 6. High-Value Customers at Risk
- 1,426 high-balance customers churned
- Average balance: 125,504
- Average age: 44.76

### 7. Active Members — Churn by Country
- Germany: 23.72% — highest even among active members
"""

with open('findings.md', 'w') as f:
    f.write(summary)

print("Saved successfully")

Saved successfully
